# CorrDiff — Fase 6 — Ciclo Diurno
Refletividade em dBZ condicionada à disponibilidade observacional.

In [ ]:
from pathlib import Path
import json, pandas as pd, matplotlib.pyplot as plt
OUT = Path('../analysis_outputs/06_diurnal')
summary = json.loads((OUT/'analysis_summary.json').read_text())
summary


## Cobertura por hora local

In [ ]:
cov = pd.read_parquet(OUT/'hourly_coverage_local.parquet')
fig, ax = plt.subplots(figsize=(10,4))
ax.bar(cov.hour_local, cov.coverage_ratio)
ax.set(xlabel='Hora local', ylabel='Cobertura', title='Cobertura temporal por hora local', ylim=(0,1))
ax.set_xticks(range(24)); plt.tight_layout(); plt.show()


## Frequência temporal por hora local

In [ ]:
events = pd.read_parquet(OUT/'hourly_event_rates_local.parquet')
for eid in ['gt_0','ge_20','ge_30','ge_40','ge_45']:
    t=events[events.event_id.eq(eid)].sort_values('hour_local')
    fig, ax = plt.subplots(figsize=(10,4)); ax.plot(t.hour_local,t.timestamp_any_event_rate,marker='o')
    ax.set(xlabel='Hora local',ylabel='P(evento | disponível, hora)',title=eid); ax.set_xticks(range(24)); plt.tight_layout(); plt.show()


## Perfil bruto × padronizado por estação

In [ ]:
std = pd.read_parquet(OUT/'standardized_hourly_event_rates_local.parquet')
for eid in ['gt_0','ge_30','ge_40','ge_45']:
    r=events[events.event_id.eq(eid)].sort_values('hour_local'); s=std[std.event_id.eq(eid)].sort_values('hour_local')
    fig, ax=plt.subplots(figsize=(10,4)); ax.plot(r.hour_local,r.timestamp_any_event_rate,marker='o',label='bruto'); ax.plot(s.hour_local,s.timestamp_any_event_rate,marker='o',label='padronizado por estação')
    ax.set(xlabel='Hora local',ylabel='P(evento | disponível, hora)',title=eid); ax.set_xticks(range(24)); ax.legend(); plt.tight_layout(); plt.show()


## Frequência × extensão espacial

In [ ]:
for eid in ['ge_30','ge_40','ge_45']:
    display(events[events.event_id.eq(eid)][['hour_local','timestamp_any_event_rate','mean_event_pixel_fraction_given_event_timestamp','mean_positive_dbz']])


## Ciclo por estação

In [ ]:
sea=pd.read_parquet(OUT/'season_hour_event_rates_local.parquet'); eid='ge_40'
fig, ax=plt.subplots(figsize=(10,4))
for sc,g in sea[sea.event_id.eq(eid)].groupby('season_code'):
    g=g.sort_values('hour_local'); ax.plot(g.hour_local,g.timestamp_any_event_rate,marker='o',label=sc)
ax.set(xlabel='Hora local',ylabel='P(evento | disponível, hora)',title=f'{eid} por estação'); ax.set_xticks(range(24)); ax.legend(); plt.tight_layout(); plt.show()


## Preditores atmosféricos

In [ ]:
p=OUT/'hourly_predictor_statistics_local.parquet'
if p.exists():
    pred=pd.read_parquet(p)
    for name in ['tcwv','r_500','t_850','t_500','wind_speed_10']:
        t=pred[pred.predictor.eq(name)].sort_values('hour_local'); fig,ax=plt.subplots(figsize=(10,4)); ax.plot(t.hour_local,t['mean'],marker='o'); ax.set(xlabel='Hora local',ylabel=name,title=f'Ciclo diurno — {name}'); ax.set_xticks(range(24)); plt.tight_layout(); plt.show()
